# Notebook 02b: Phân tích Chuyên sâu Câu hỏi Q2 & Q5 (Biometrics & Herd Attack Rate)
**Đề tài:** Phân tích Sự cố Bất lợi Thuốc Thú y (OpenFDA Animal & Veterinary Adverse Events)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Thành viên thực hiện:** Trần Ngô Uyên Nhi (M2) — Nhánh Git: `feature/nhi`  
**Phạm vi sở hữu:** **Câu hỏi Q2** & **Câu hỏi Q5** (Non-ML Questions)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_processed
from src.preprocessing import log_row_count_change
from src.visualization import set_project_plot_style
set_project_plot_style()

In [ ]:
# Tải dữ liệu đã qua tiền xử lý chuẩn hóa từ data/processed/
data = load_processed(PROJECT_ROOT / 'data' / 'processed')
reports = data['reports']
drugs = data['drugs']
reactions = data['reactions']
outcomes = data['outcomes']

print(f"[Data Loaded] Reports: {reports.shape}, Drugs: {drugs.shape}, Reactions: {reactions.shape}, Outcomes: {outcomes.shape}")

---
## 1. Câu hỏi Q2: Đặc điểm sinh học & Nguy cơ xảy ra Biến cố Nghiêm trọng

### Phát biểu câu hỏi & Động lực (Question & Motivation)
- **Phát biểu câu hỏi:**  
  > **"Những đặc điểm sinh học nào (giai đoạn sống, trọng lượng cơ thể, giới tính) có liên quan đến các biến cố bất lợi nghiêm trọng, sau khi đã kiểm soát yếu tố loài và số lượng thuốc sử dụng?"**
- **Người quan tâm & Quyết định hỗ trợ (Target Users & Decisions):**
  - **Bác sĩ thú y lâm sàng:** Cần xác định nhóm vật nuôi có nguy cơ cao gặp biến cố nghiêm trọng (thú già/thú non, nhẹ cân, dùng đa trị liệu) để tối ưu liều dùng và theo dõi sau dùng thuốc.
  - **Hệ thống cảnh giác dược:** Cung cấp căn cứ đưa cảnh báo thận trọng vào tờ hướng dẫn sử dụng thuốc đối với các phân khúc sinh học nhạy cảm.
- **Tiêu chí độ sâu áp dụng (Mục 3.3):**
  - `D3` (Derived variables): Chuẩn hóa `age_years`, `weight_kg`, phân nhóm giai đoạn sống `life_stage`, và số thuốc đồng thời `num_drugs`.
  - `D4` (Controlling for a confounder): Kiểm soát đồng thời biến loài (Chó vs Mèo) và mức độ đa trị liệu bằng phân tầng (stratification) hoặc hồi quy đa biến.
  - `D5` (Quantifying uncertainty): Ước lượng Adjusted Odds Ratio (aOR) kèm khoảng tin cậy 95% và kiểm định giả thuyết thống kê.
  - `D6` (Robustness check): Kiểm tra độ vững của kết luận khi thay đổi ngưỡng cắt ngoại lai tuổi/cân nặng và khi loại trừ nhóm `'Unknown'`.

### 1.1. Tiền xử lý cho Q2 (Preprocessing & Row Count Tracking)
*(Mô tả các bước xử lý dữ liệu sinh trắc học cá thể, lọc bản ghi hợp lệ và theo dõi biến động số dòng)*

In [ ]:
# TODO [Nhi - M2]: Triển khai tiền xử lý cho Q2 kèm log_row_count_change

### 1.2. Phương pháp phân tích (Analysis Method)
*(Giải thích mô hình hồi quy Logistic / phân tầng Cochran-Mantel-Haenszel để kiểm soát biến nhiễu)*

In [ ]:
# TODO [Nhi - M2]: Triển khai phân tích Q2 (Ước lượng tỷ suất chênh điều chỉnh aOR, CI 95%, kiểm tra độ vững D6)

### 1.3. Kết quả & Trực quan hóa (Results & Visualizations)
*(Vẽ tối thiểu 2 biểu đồ: Tỷ lệ biến cố nghiêm trọng theo giai đoạn sống & cân nặng sau khi phân tầng loài; Forest plot hiển thị aOR và CI 95%)*

In [ ]:
# TODO [Nhi - M2]: Vẽ >= 2 biểu đồ chuẩn mực cho Q2

### 1.4. Trả lời Câu hỏi & Thảo luận (Answer & Practical Meaning)
*(Trả lời bằng số liệu định lượng cụ thể, thảo luận ý nghĩa thực tiễn, phân tích bất ngờ và giới hạn)*

---
## 2. Câu hỏi Q5: Các yếu tố liên quan đến Tỷ lệ Động vật Bị ảnh hưởng ở Đàn (Herd Attack Rate)

### Phát biểu câu hỏi & Động lực (Question & Motivation)
- **Phát biểu câu hỏi:**  
  > **"Những yếu tố nào có liên quan đến tỷ lệ động vật bị ảnh hưởng trong tổng số động vật được điều trị ở mỗi đàn?"**
- **Biến mục tiêu:**  
  $$\text{attack\_rate} = \frac{\text{number\_of\_animals\_affected}}{\text{number\_of\_animals\_treated}} \quad (\text{xét đàn có } \text{treated} > 0)$$
- **Người quan tâm & Quyết định hỗ trợ (Target Users & Decisions):**
  - **Bác sĩ thú y trang trại & Chủ trang trại chăn nuôi:** Cần biết yếu tố nào (loài gia súc: Bò, Heo, Gà; đường dùng thuốc: tiêm, pha nước uống, trộn thức ăn; quy mô đàn) làm trầm trọng hóa mức độ lan rộng của sự cố để xây dựng phương án phòng ngừa.
  - **Cơ quan quản lý an toàn dịch tễ:** Giám sát rủi ro ngộ độc hoặc phản ứng phụ diện rộng trong chăn nuôi công nghiệp.
- **Tiêu chí độ sâu áp dụng (Mục 3.3):**
  - `D1` (Combining data): Nối thông tin báo cáo đàn `reports` với bảng thông tin thuốc `drugs` (đường dùng, nhóm thuốc).
  - `D3` (Derived variables): Tính toán biến tỷ lệ $\text{attack\_rate} \in [0, 1]$ và phân nhóm quy mô đàn (`herd_size_group`).
  - `D4` (Controlling for a confounder): Kiểm soát loài đàn (Bò vs Loài khác) và phân loại quy mô đàn khi đánh giá tác động của đường dùng thuốc.
  - `D5` (Quantifying uncertainty): Kiểm định Kruskal-Wallis / Mann-Whitney U, Bootstrap CI 95% cho trung vị tỷ lệ ảnh hưởng.
  - `D6` (Robustness check): So sánh kết luận phân tích khi loại trừ 13 ca ngoại lai có $\text{attack\_rate} > 1$ (do lỗi nhập liệu gốc) so với khi đưa về chặn trên 1.0.

### 2.1. Tiền xử lý cho Q5 (Preprocessing & Row Count Tracking)
*(Lọc các ca báo cáo đàn có `treated > 0`, kết nối đường dùng thuốc từ `drugs`, theo dõi số lượng dòng)*

In [ ]:
# TODO [Nhi - M2]: Triển khai tiền xử lý cho Q5 kèm log_row_count_change

### 2.2. Phương pháp phân tích (Analysis Method)
*(Phân tích phi tham số, ước lượng phân vị, so sánh nhóm theo đường dùng thuốc và quy mô đàn)*

In [ ]:
# TODO [Nhi - M2]: Triển khai phân tích Q5 (Kruskal-Wallis, Bootstrap CI, Robustness D6)

### 2.3. Kết quả & Trực quan hóa (Results & Visualizations)
*(Vẽ tối thiểu 2 biểu đồ: Boxplot/Violinplot phân bố attack_rate theo đường dùng thuốc; Biểu đồ kiểm tra độ vững D6)*

In [ ]:
# TODO [Nhi - M2]: Vẽ >= 2 biểu đồ chuẩn mực cho Q5

### 2.4. Trả lời Câu hỏi & Thảo luận (Answer & Practical Meaning)
*(Trả lời bằng số liệu cụ thể về attack_rate theo từng nhóm, ý nghĩa quản lý đàn trang trại và hạn chế)*